In [ ]:
#! pip install neo4j
#! pip install configparser

In [ ]:
from neo4j import GraphDatabase, RoutingControl
import configparser

config = configparser.ConfigParser()
config.read("neo4j_config.ini")
db_config = config['db_server']
db_url = db_config['DB_URL']
db_user = db_config['DB_USER']
db_pass = db_config['DB_PASS']
db_name = db_config['DB_NAME']
driver = GraphDatabase.driver(db_url, auth=(db_user, db_pass))

# confirm the DB is connected
with driver.session() as session:
    result = session.run("RETURN 1")
    if result.single()[0] == 1:
        print("Connected!")
    else:
        print("Failed to connect!")
 
schema_statement = [
    "CREATE CONSTRAINT con1 IF NOT EXISTS FOR (seq:Sequence) REQUIRE seq.sequence_token IS UNIQUE;",
    "CREATE CONSTRAINT con2 IF NOT EXISTS FOR (s:Sample) REQUIRE s.sample_token IS UNIQUE;",
    "CREATE INDEX ind1 IF NOT EXISTS FOR (seq:Sequence) ON (seq.sequence_token);",
    "CREATE INDEX ind2 IF NOT EXISTS FOR (s:Sample) ON (s.sample_token);"
]

with driver.session(database=db_name) as session:
    for statement in schema_statements:
        session.run(statement)

# Load nuScenes data

## Create scene nodes

In [ ]:
import json
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-mini', dataroot='/data/sets/nuscenes', verbose=True)
all_sces = nusc.scene

records, summary, keys = driver.execute.query(
    '''
        UNWIND $rows AS row
        MERGE (s:Sequence {sequence_token:row.token, sequence_name:row.name, nbr_smaples:row.nbr_samples, first_sample_token:row.first_sample_token, last_sample_token:row.last_sample_token})
    ''',
    databse_=db_name, routing_=RoutingControl.WRITE, rows=all_sces
)

## Create sample nodes in each scene

In [ ]:
def add_sample_node(session, sample):
    create_sample_node = ("MERGE (s:Sample {sample_token:$sample_token, timestamp:$timestamp, sequence_token:$sequence_token }) ")
    sample_node = session.run(create_sample_node, 
                              sample_token=sample['token'], timestamp=sample['timestamp'], sequence_token=sample['scene_token'])
    # connect sample to scene
    connect_sample_seq = ("MATCH (seq_node:Sequence{sequence_token:$sequence_token}) "
                          "MATCH (sample_node:Sample{sample_token:$sample_token}) "
                          "MERGE (seq_node)-[r:CONTAINS]->(sample_node) ")
    edge = session.run(connect_sample_seq, sequence_token=sample['scene_token'], sample_token=sample['token'])
    # update sample with prev and next sample info
    update_node = ("MATCH (s:Sample{sample_token:$sample_token}) SET s.next_sample=$next_sample RETURN s")
    update_return = session.run(update_node, sample_token=sample['token'], next_sample=sample['next'])
    update_node = ("MATCH (s:Sample{sample_token:$sample_token}) SET s.prev_sample=$prev_sample RETURN s")
    update_return = session.run(update_node, sample_token=sample['token'], prev_sample=sample['prev'])
    # connect with prev sample
    if sample['prev'] != '':
        connect_samples = ("MATCH (s1:Sample{sample_token:$sample_token}) "
                           "MATCH (s0:Sample{sample_token:$prev_sample}) "
                           "MERGE (s1)-[r:PREV]->[s0] ")
        edge = session.run(connect_samples, sample_token=sample['token'], prev_sample=sample['prev'])
    
with driver.session(database=db_name) as session:
    for seq in all_sces:
        print('Current scene: ', seq.token)
        first_sample_token = seq['first_sample_token']
        next_sample = first_sample_token
        while next_sample != '':
            my_sample = nusc.get('sample', next_sample)
            add_sample_node(my_sample)
            next_sample = nuse.get('sample', my_sample['next'])